In [1]:
# https://github.com/carpenter-singh-lab/2022_Haghighi_NatureMethods/blob/main/1.dataset_curation/curate_dataset.py

In [2]:
import pandas as pd
from pathlib import Path
from IPython.display import display

In [3]:
# First download the data from
# s3://cellpainting-gallery/cpg0003-rosetta/broad/workspace/preprocessed_data
# aws s3 sync \
#  --no-sign-request \
#  s3://cellpainting-gallery/cpg0003-rosetta/broad/workspace/preprocessed_data .
# and save it in the current folder
dataset_paths = {
    "LINCS-Pilot1": {
        "l1k": "LINCS-Pilot1/L1000/replicate_level_l1k.csv.gz",
        "cp": "LINCS-Pilot1/CellPainting/replicate_level_cp_augmented.csv.gz",
    },
    "CDRP-BBBC047-Bray": {
        "l1k": "CDRP-BBBC047-Bray/L1000/replicate_level_l1k.csv.gz",
        "cp": "CDRP-BBBC047-Bray/CellPainting/replicate_level_cp_augmented.csv.gz",
    },
    "TA-ORF-BBBC037-Rohban": {
        "l1k": "TA-ORF-BBBC037-Rohban/L1000/replicate_level_l1k.csv.gz",
        "cp": "TA-ORF-BBBC037-Rohban/CellPainting/replicate_level_cp_augmented.csv.gz",
    },
    "LUAD-BBBC041-Caicedo": {
        "l1k": "LUAD-BBBC041-Caicedo/L1000/replicate_level_l1k.csv.gz",
        "cp": "LUAD-BBBC041-Caicedo/CellPainting/replicate_level_cp_augmented.csv.gz",
    },
}

In [4]:
# Define column mappings for each dataset and data type
column_rename_mappings = {
    "CDRP-BBBC047-Bray": {
        "l1k": {
            "pert_id": "Metadata_pert_id",
            "pert_dose": "Metadata_pert_dose_micromolar",
            "det_plate": "Metadata_Plate",
            "CPD_NAME": "Metadata_pert_iname",
            "CPD_TYPE": "Metadata_cdrp_group",
            "CPD_SMILES": "Metadata_SMILES",
        },
        "cp": {
            "Metadata_broad_sample": "Metadata_pert_id",
            "Metadata_broad_sample_type": "Metadata_pert_type",
            "Metadata_mmoles_per_liter2": "Metadata_pert_dose_micromolar",
        },
    },
    "LINCS-Pilot1": {
        "l1k": {
            "pert_dose": "Metadata_pert_dose_micromolar",
            "det_plate": "Metadata_Plate",
            "cell_id": "Metadata_cell_id",
            "det_well": "Metadata_Well",
            "mfc_plate_name": "Metadata_ARP_ID",
            "pert_iname_x": "Metadata_pert_iname",
            "pert_time": "Metadata_pert_timepoint",
            "pert_mfc_id": "Metadata_pert_id",
            "pert_type_x": "Metadata_pert_type",
            "x_smiles": "Metadata_SMILES",
        },
        "cp": {
            "Metadata_broad_sample": "Metadata_pert_id",
            "Metadata_broad_sample_type": "Metadata_pert_type",
            "Metadata_mmoles_per_liter": "Metadata_pert_dose_micromolar",
            "pert_iname": "Metadata_pert_iname",
        },
    },
    "TA-ORF-BBBC037-Rohban": {
        "l1k": {
            "det_plate": "Metadata_Plate",
            "cell_id": "Metadata_cell_id",
            "det_well": "Metadata_Well",
            "mfc_plate_name": "Metadata_ARP_ID",
            "pert_time": "Metadata_pert_timepoint",
            "pert_mfc_id": "Metadata_pert_id",
            "pert_type": "Metadata_pert_type",
            "x_genesymbol_mutation": "Metadata_genesymbol_mutation",
        },
        "cp": {
            "Metadata_broad_sample": "Metadata_pert_id",
            "Metadata_broad_sample_type": "Metadata_pert_type",
            "Metadata_pert_name": "Metadata_genesymbol_mutation",
            "Metadata_gene_name": "Metadata_genesymbol",
        },
    },
    "LUAD-BBBC041-Caicedo": {
        "l1k": {
            "det_plate": "Metadata_Plate",
            "cell_id": "Metadata_cell_id",
            "det_well": "Metadata_Well",
            "mfc_plate_name": "Metadata_ARP_ID",
            "pert_time": "Metadata_pert_timepoint",
            "pert_mfc_id": "Metadata_pert_id",
            "pert_type": "Metadata_pert_type",
            "x_transcriptdb": "Metadata_transcriptdb",
        },
        "cp": {
            "Metadata_broad_sample": "Metadata_pert_id",
            "Metadata_broad_sample_type": "Metadata_pert_type",
            "x_mutation_status": "Metadata_genesymbol_mutation",
            "Symbol": "Metadata_genesymbol",
        },
    },
}

In [5]:
# Define the columns we want to keep for each dataset and data type
columns_to_keep = {
    "CDRP-BBBC047-Bray": {
        "l1k": [
            "Metadata_Plate",
            "Metadata_pert_id",
            "Metadata_pert_iname",
            "Metadata_pert_dose_micromolar",
            "Metadata_cdrp_group",
            "Metadata_SMILES",
        ],
        "cp": [
            "Metadata_Plate_Map_Name",
            "Metadata_Plate",
            "Metadata_Well",
            "Metadata_pert_id",
            "Metadata_pert_dose_micromolar",
            "Metadata_pert_type",
            "Metadata_cell_id",
        ],
    },
    "LINCS-Pilot1": {
        "l1k": [
            "Metadata_Plate",
            "Metadata_Well",
            "Metadata_pert_id",
            "Metadata_pert_type",
            "Metadata_pert_dose_micromolar",
            "Metadata_cell_id",
            "Metadata_pert_iname",
            "Metadata_ARP_ID",
            "Metadata_pert_timepoint",
            "Metadata_SMILES",
        ],
        "cp": [
            "Metadata_Plate_Map_Name",
            "Metadata_Plate",
            "Metadata_Well",
            "Metadata_pert_id",
            "Metadata_pert_type",
            "Metadata_pert_dose_micromolar",
            "Metadata_cell_id",
            "Metadata_pert_iname",
        ],
    },
    "TA-ORF-BBBC037-Rohban": {
        "l1k": [
            "Metadata_Plate",
            "Metadata_Well",
            "Metadata_pert_id",
            "Metadata_pert_type",
            "Metadata_cell_id",
            "Metadata_ARP_ID",
            "Metadata_pert_timepoint",
            "Metadata_genesymbol_mutation",
        ],
        "cp": [
            "Metadata_Plate_Map_Name",
            "Metadata_Plate",
            "Metadata_Well",
            "Metadata_pert_id",
            "Metadata_pert_type",
            "Metadata_cell_id",
            "Metadata_genesymbol_mutation",
            "Metadata_genesymbol",
        ],
    },
    "LUAD-BBBC041-Caicedo": {
        "l1k": [
            "Metadata_Plate",
            "Metadata_Well",
            "Metadata_pert_id",
            "Metadata_pert_type",
            "Metadata_cell_id",
            "Metadata_ARP_ID",
            "Metadata_pert_timepoint",
            "Metadata_transcriptdb",
        ],
        "cp": [
            "Metadata_Plate_Map_Name",
            "Metadata_Plate",
            "Metadata_Well",
            "Metadata_pert_id",
            "Metadata_pert_type",
            "Metadata_cell_id",
            "Metadata_genesymbol_mutation",
            "Metadata_genesymbol",
        ],
    },
}

In [6]:
# First load the data
dataset_data = {}
for dataset_name, paths in dataset_paths.items():
    dataset_data[dataset_name] = {}
    for data_type, dataset_path in paths.items():
        parquet_path = dataset_path.replace(".csv.gz", ".parquet")
        if not Path(parquet_path).exists():
            data = pd.read_csv(dataset_path, low_memory=False)
            data.to_parquet(parquet_path)
            dataset_data[dataset_name][data_type] = data
        else:
            data = pd.read_parquet(parquet_path)
            dataset_data[dataset_name][data_type] = data

In [7]:
# Then apply the column renaming
for dataset_name, data_types in dataset_data.items():
    for data_type, data in data_types.items():
        if (
            dataset_name in column_rename_mappings
            and data_type in column_rename_mappings[dataset_name]
        ):
            # First, identify feature columns we want to preserve
            if data_type == "l1k":
                feature_mask = data.columns.str.endswith("_at")
            else:  # cp
                feature_mask = (
                    data.columns.str.startswith("Cells_")
                    | data.columns.str.startswith("Cytoplasm_")
                    | data.columns.str.startswith("Nuclei_")
                )
            feature_cols = data.columns[feature_mask]
            metadata_cols = data.columns[~feature_mask]

            # Apply renaming only to metadata columns
            rename_mapping = {
                k: v
                for k, v in column_rename_mappings[dataset_name][data_type].items()
                if k in metadata_cols
            }

            # Check if new name already exists and drop it if so
            for old, new in rename_mapping.items():
                if new in data.columns and new != old:
                    data.drop(columns=[new], inplace=True)
            # Rename metadata columns
            data = data.rename(columns=rename_mapping)

            # Keep only desired metadata columns plus all feature columns
            keep_metadata = columns_to_keep[dataset_name][data_type]
            dataset_data[dataset_name][data_type] = data[
                keep_metadata + feature_cols.tolist()
            ]

In [8]:
# Make "Metadata_Well" uppercase
for dataset_name, data_types in dataset_data.items():
    for data_type, data in data_types.items():
        if "Metadata_Well" in data.columns:
            data["Metadata_Well"] = data["Metadata_Well"].str.upper()

In [9]:
# Make "Metadata_cell_id" = U2OS for CDRP-BBBC047-Bray cp
dataset_data["CDRP-BBBC047-Bray"]["l1k"]["Metadata_cell_id"] = "U2OS"

In [10]:
# Set timepoints
dataset_data["LINCS-Pilot1"]["cp"]["Metadata_pert_timepoint"] = 48
dataset_data["LINCS-Pilot1"]["l1k"]["Metadata_pert_timepoint"] = 24

dataset_data["CDRP-BBBC047-Bray"]["cp"]["Metadata_pert_timepoint"] = 48
dataset_data["CDRP-BBBC047-Bray"]["l1k"]["Metadata_pert_timepoint"] = 6

dataset_data["TA-ORF-BBBC037-Rohban"]["cp"]["Metadata_pert_timepoint"] = 72
dataset_data["TA-ORF-BBBC037-Rohban"]["l1k"]["Metadata_pert_timepoint"] = 72

dataset_data["LUAD-BBBC041-Caicedo"]["cp"]["Metadata_pert_timepoint"] = 96
dataset_data["LUAD-BBBC041-Caicedo"]["l1k"]["Metadata_pert_timepoint"] = 96

In [11]:
# Display the datasets
for dataset_name, data_types in dataset_data.items():
    for data_type, data in data_types.items():
        display(f"Dataset: {dataset_name}, Data Type: {data_type}")
        display(data.sample(5)[data.columns[data.columns.str.startswith("Metadata")]])

'Dataset: LINCS-Pilot1, Data Type: l1k'

,Metadata_Plate,Metadata_Well,Metadata_pert_id,Metadata_pert_type,Metadata_pert_dose_micromolar,Metadata_cell_id,Metadata_pert_iname,Metadata_ARP_ID,Metadata_pert_timepoint,Metadata_SMILES
25134,REP.A026_A549_24H_X2_B29,L07,BRD-K91825936-001-01-2,trt_cp,10.00000,A549,zk811752,AB00016191,24,C[C@@H]1CN(Cc2ccc(F)cc2)CCN1C(=O)COc1ccc(Cl)cc...
25463,REP.A026_A549_24H_X3_B29,J13,-666,ctl_vehicle,-666.00000,A549,DMSO,AB00016191,24,-666
2349,REP.A003_A549_24H_X1_B27,F20,BRD-K32821942-001-21-3,trt_cp,3.33333,A549,azathioprine,AB00016886,24,Cn1cnc(c1Sc1[nH]cnc2ncnc12)[N+]([O-])=O
7720,REP.A010_A549_24H_X3_B32,L08,BRD-K14653796-001-01-1,trt_cp,3.33333,A549,tasquinimod,AB00016881,24,COc1cccc2n(C)c(=O)c(C(=O)N(C)c3ccc(cc3)C(F)(F)...
19080,REP.A020_A549_24H_X3_B29,F11,-666,ctl_vehicle,-666.00000,A549,DMSO,AB00016188,24,-666


'Dataset: LINCS-Pilot1, Data Type: cp'

,Metadata_Plate_Map_Name,Metadata_Plate,Metadata_Well,Metadata_pert_id,Metadata_pert_type,Metadata_pert_dose_micromolar,Metadata_cell_id,Metadata_pert_iname,Metadata_pert_timepoint
45757,C-7161-01-LM6-014,SQ00015233,C15,BRD-K57252450-001-02-5,trt,1.11110,A549,cilengitide,48
29780,C-7161-01-LM6-010,SQ00015217,I22,BRD-K14175878-001-01-6,trt,0.37037,A549,MK-0812,48
29497,C-7161-01-LM6-010,SQ00015216,N03,BRD-A14941520-051-01-0,trt,1.11110,A549,dofequidar,48
37196,C-7161-01-LM6-003,SQ00015127,N22,BRD-K35775715-001-03-1,trt,0.37037,A549,liranaftate,48
46860,C-7161-01-LM6-028,SQ00015058,A14,BRD-K92441787-001-04-1,trt,3.33330,A549,bexarotene,48


'Dataset: CDRP-BBBC047-Bray, Data Type: l1k'

...

,Metadata_Plate_Map_Name,Metadata_Plate,Metadata_Well,Metadata_pert_id,Metadata_pert_type,Metadata_cell_id,Metadata_genesymbol_mutation,Metadata_genesymbol,Metadata_pert_timepoint
55,TAORF_REFERENCE_SET,41744,C08,ccsbBroad304_11052,trt,U2OS,PPP2R5C_WT.1,PPP2R5C,72
439,TAORF_REFERENCE_SET,41754,C08,ccsbBroad304_11052,trt,U2OS,PPP2R5C_WT.1,PPP2R5C,72
57,TAORF_REFERENCE_SET,41744,C10,ccsbBroad304_11095,trt,U2OS,RELA_WT.1,RELA,72
1578,TAORF_REFERENCE_SET,41757,B19,BRDN0000464904,trt,U2OS,EIF4E_WT.1,EIF4E,72
111,TAORF_REFERENCE_SET,41744,E16,negcon,trt,U2OS,EMPTY_,EMPTY,72


'Dataset: LUAD-BBBC041-Caicedo, Data Type: l1k'

,Metadata_Plate,Metadata_Well,Metadata_pert_id,Metadata_pert_type,Metadata_cell_id,Metadata_ARP_ID,Metadata_pert_timepoint,Metadata_transcriptdb
4008,TA.OE015_A549_96H_X1_B19,G23,TRCN0000471571,trt,A549,DOC49.50.51.52.A,96,NM_004365.3
412,TA.OE014_A549_96H_X5_B19,A18,TRCN0000477544,trt,A549,DOC45.46.47.48.A,96,NM_002863.4
3342,TA.OE015_A549_96H_X7_B19,G06,BRDN0000560220,trt,A549,DOC49.50.51.52.A,96,NM_014225.5
2940,TA.OE014_A549_96H_X5_B19,M09,BRDN0000553247,trt,A549,DOC45.46.47.48.A,96,NM_012289.3:c.330G>A
2687,TA.OE015_A549_96H_X8_B19,I08,BRDN0000552857,trt,A549,DOC49.50.51.52.A,96,NM_001126112.2:c.832C>G


'Dataset: LUAD-BBBC041-Caicedo, Data Type: cp'

,Metadata_Plate_Map_Name,Metadata_Plate,Metadata_Well,Metadata_pert_id,Metadata_pert_type,Metadata_cell_id,Metadata_genesymbol_mutation,Metadata_genesymbol,Metadata_pert_timepoint
951,DOC45.46.47.48,52664,E23,BRDN0000561664,trt,A549,TP53_WT.c,TP53,96
1450,DOC45.46.47.48,52653,H14,BRDN0000560696,trt,A549,ARAF_p.D429A,ARAF,96
4565,DOC49.50.51.52,52654,H19,EMPTY,control,A549,negcon,NaN,96
5285,DOC49.50.51.52,52654,L13,BRDN0000554171,trt,A549,PIK3CA_p.123_124MP>IA,PIK3CA,96
1050,DOC45.46.47.48,52653,F12,BRDN0000553513,trt,A549,EGFR_p.Q1020H,EGFR,96


In [12]:
# TA-ORF-BBBC037-Rohban cp does not correctly identify Metadata_pert_type, because it marks all as trt.
for dataset_name, data_types in dataset_data.items():
    for data_type, data in data_types.items():
        if "Metadata_pert_type" in data.columns:
            data["Metadata_pert_type"] = data["Metadata_pert_type"].replace(
                {"ctl_vehicle": "control", "trt_cp": "trt"}
            )

In [13]:
# Print columns for each dataset and data type
print("\nColumns in each dataset:")
for dataset_name, data_types in dataset_data.items():
    print(f"\n{dataset_name}:")
    for data_type, data in data_types.items():
        metadata_cols = [col for col in data.columns if col.startswith("Metadata")]
        print(f"  {data_type}: {sorted(metadata_cols)}")


Columns in each dataset:

LINCS-Pilot1:
  l1k: ['Metadata_ARP_ID', 'Metadata_Plate', 'Metadata_SMILES', 'Metadata_Well', 'Metadata_cell_id', 'Metadata_pert_dose_micromolar', 'Metadata_pert_id', 'Metadata_pert_iname', 'Metadata_pert_timepoint', 'Metadata_pert_type']
  cp: ['Metadata_Plate', 'Metadata_Plate_Map_Name', 'Metadata_Well', 'Metadata_cell_id', 'Metadata_pert_dose_micromolar', 'Metadata_pert_id', 'Metadata_pert_iname', 'Metadata_pert_timepoint', 'Metadata_pert_type']

CDRP-BBBC047-Bray:
  l1k: ['Metadata_Plate', 'Metadata_SMILES', 'Metadata_cdrp_group', 'Metadata_cell_id', 'Metadata_pert_dose_micromolar', 'Metadata_pert_id', 'Metadata_pert_iname', 'Metadata_pert_timepoint']
  cp: ['Metadata_Plate', 'Metadata_Plate_Map_Name', 'Metadata_Well', 'Metadata_cell_id', 'Metadata_pert_dose_micromolar', 'Metadata_pert_id', 'Metadata_pert_timepoint', 'Metadata_pert_type']

TA-ORF-BBBC037-Rohban:
  l1k: ['Metadata_ARP_ID', 'Metadata_Plate', 'Metadata_Well', 'Metadata_cell_id', 'Metadata_g

In [14]:
# Find common columns between l1k datasets
l1k_common = set.intersection(
    *[set(data_types["l1k"].columns) for data_types in dataset_data.values()]
)
l1k_metadata_common = sorted([col for col in l1k_common if col.startswith("Metadata")])

In [15]:
# Find common columns between cp datasets
cp_common = set.intersection(
    *[set(data_types["cp"].columns) for data_types in dataset_data.values()]
)
cp_metadata_common = sorted([col for col in cp_common if col.startswith("Metadata")])

In [16]:
# Find common columns across all datasets
all_common = set.intersection(l1k_common, cp_common)
all_metadata_common = sorted([col for col in all_common if col.startswith("Metadata")])

In [17]:
print("\nCommon Metadata columns across L1K datasets:")
print(l1k_metadata_common)
print("\nCommon Metadata columns across CP datasets:")
print(cp_metadata_common)
print("\nCommon Metadata columns across ALL datasets:")
print(all_metadata_common)


Common Metadata columns across L1K datasets:
['Metadata_Plate', 'Metadata_cell_id', 'Metadata_pert_id', 'Metadata_pert_timepoint']

Common Metadata columns across CP datasets:
['Metadata_Plate', 'Metadata_Plate_Map_Name', 'Metadata_Well', 'Metadata_cell_id', 'Metadata_pert_id', 'Metadata_pert_timepoint', 'Metadata_pert_type']

Common Metadata columns across ALL datasets:
['Metadata_Plate', 'Metadata_cell_id', 'Metadata_pert_id', 'Metadata_pert_timepoint']


In [18]:
# Check for duplicate columns within each dataset
for dataset_name, data_types in dataset_data.items():
    for data_type, data in data_types.items():
        duplicate_cols = data.columns.duplicated()
        if any(duplicate_cols):
            print(f"Duplicate columns found in {dataset_name} {data_type}:")
            print(data.columns[duplicate_cols])

In [19]:
# Create markdown output for datasets
markdown_output = "# Dataset Samples\n\n"

for dataset_name, data_types in dataset_data.items():
    markdown_output += f"## {dataset_name}\n\n"
    for data_type, data in data_types.items():
        markdown_output += f"### {data_type.upper()} Data\n\n"
        # Convert sample to markdown table
        sample_df = data.sample(5)[
            [col for col in data.columns if col.startswith("Metadata")]
        ]
        markdown_output += sample_df.to_markdown(index=False) + "\n\n"
        display(sample_df.head())

,Metadata_Plate,Metadata_Well,Metadata_pert_id,Metadata_pert_type,Metadata_pert_dose_micromolar,Metadata_cell_id,Metadata_pert_iname,Metadata_ARP_ID,Metadata_pert_timepoint,Metadata_SMILES
10687,REP.A013_A549_24H_X1_B29,L22,BRD-K42679050-001-02-1,trt,0.370370,A549,Y-27152,AB00016860,24,CC(=O)N(OCc1ccccc1)[C@H]1[C@H](O)C(C)(C)Oc2ccc...
17906,REP.A019_A549_24H_X3_B29,D12,BRD-K45252063-001-13-6,trt,0.041152,A549,clofibrate,AB00016189,24,CCOC(=O)C(C)(C)Oc1ccc(Cl)cc1
13986,REP.A016_A549_24H_X1_B32,K14,BRD-K55013654-003-02-3,trt,3.333330,A549,lorcaserin,AB00018081,24,C[C@H]1CNCCc2ccc(Cl)cc12
9877,REP.A012_A549_24H_X3_B25,H19,BRD-K01815685-001-15-5,trt,10.000000,A549,indole,AB00016883,24,OCc1c[nH]c2ccccc12
18588,REP.A020_A549_24H_X2_B25,A08,BRD-K82522873-001-01-1,trt,3.333330,A549,n6022,AB00016188,24,Cc1cc(ccc1-n1c(CCC(O)=O)ccc1-c1ccc(cc1)-n1ccnc...


,Metadata_Plate_Map_Name,Metadata_Plate,Metadata_Well,Metadata_pert_id,Metadata_pert_type,Metadata_pert_dose_micromolar,Metadata_cell_id,Metadata_pert_iname,Metadata_pert_timepoint
6296,C-7161-01-LM6-019,SQ00015150,G09,BRD-K04887706-375-01-4,trt,1.111100,A549,NaN,48
27354,C-7161-01-LM6-004,SQ00015159,D20,BRD-K20285085-001-07-1,trt,3.333300,A549,R406,48
15761,C-7161-01-LM6-021,SQ00015100,A19,BRD-K19111024-001-20-9,trt,10.000000,A549,clofibric-acid,48
41368,C-7161-01-LM6-016,SQ00015138,L18,BRD-K55026842-001-01-4,trt,0.041152,A549,azacitidine,48
32995,C-7161-01-LM6-012,SQ00015196,O21,BRD-K03390685-001-01-7,trt,1.111100,A549,cobimetinib,48


,Metadata_Plate,Metadata_pert_id,Metadata_pert_iname,Metadata_pert_dose_micromolar,Metadata_cdrp_group,Metadata_SMILES,Metadata_cell_id,Metadata_pert_timepoint
56715,PAC058_U2OS_6H_X3_B1_UNI4445L,BRD-K35243934-001-01-9,BRD-K35243934,10.1718,DOS,COC(=O)C[C@H]1CC[C@@H]2[C@H](COC[C@H](O)CN2S(=...,U2OS,6
21678,PAC025_U2OS_6H_X1_B1_UNI4445R,BRD-K61505971-001-01-2,BRD-K61505971,10.0182,DOS,C\C=C\c1cnc2O[C@@H](CN(C)S(=O)(=O)c3ccccc3F)[C...,U2OS,6
9012,PAC012_U2OS_6H_X3_B1_UNI4445R,BRD-K02201521-001-01-5,BRD-K02201521,10.1782,DOS,COCC(=O)N(C)C[C@@H]1Oc2ncc(cc2C(=O)N(C[C@@H]1C...,U2OS,6
31425,PAC034_U2OS_6H_X2_B1_UNI4445R,BRD-K06724886-001-01-1,BRD-K06724886,10.0556,DOS,CO[C@H]1CN(C)C(=O)c2cc(NC(=O)C3CCC3)ccc2OC[C@@...,U2OS,6
53132,PAC055_U2OS_6H_X1_B1_UNI4445R,BRD-K45303727-001-01-0,BRD-K45303727,10.5642,DOS,C[C@@H](NC(=O)C[C@H]1O[C@@H](CO)[C@@H](NS(=O)(...,U2OS,6


,Metadata_Plate_Map_Name,Metadata_Plate,Metadata_Well,Metadata_pert_id,Metadata_pert_dose_micromolar,Metadata_pert_type,Metadata_cell_id,Metadata_pert_timepoint
20064,H-CBLB-005-4,24592,E02,DMSO,0.00,control,U2OS,48
100956,H-BIOA-001-3,25937,M11,BRD-A35511923-019-01-6,10.00,trt,U2OS,48
95040,H-CBLG-002-4,25903,F23,BRD-K11401851-001-01-0,9.90,trt,U2OS,48
91734,H-CBLF-002-4,25856,M05,BRD-K61929092-001-01-8,10.23,trt,U2OS,48
90761,H-CBLF-002-4,25854,D16,DMSO,0.00,control,U2OS,48


,Metadata_Plate,Metadata_Well,Metadata_pert_id,Metadata_pert_type,Metadata_cell_id,Metadata_ARP_ID,Metadata_pert_timepoint,Metadata_genesymbol_mutation
681,TA.OE005_U2OS_72H_X2.A2_B18,N19,BRDN0000464966,trt,U2OS,DOA60.61.62.63.A,72,MYD88_L265P
630,TA.OE005_U2OS_72H_X2.A2_B18,L14,BRDN0000464862,trt,U2OS,DOA60.61.62.63.A,72,AKT1S1
224,TA.OE005_U2OS_72H_X1_B15,J20,BRDN0000459574,trt,U2OS,DOA60.61.62.63.A,72,MOS
264,TA.OE005_U2OS_72H_X1_B15,L13,BRDN0000464954,trt,U2OS,DOA60.61.62.63.A,72,BRAF
671,TA.OE005_U2OS_72H_X2.A2_B18,N09,BRDN0000464961,trt,U2OS,DOA60.61.62.63.A,72,PRKCZ_del1-238


,Metadata_Plate_Map_Name,Metadata_Plate,Metadata_Well,Metadata_pert_id,Metadata_pert_type,Metadata_cell_id,Metadata_genesymbol_mutation,Metadata_genesymbol,Metadata_pert_timepoint
1389,TAORF_REFERENCE_SET,41756,J22,ccsbBroad304_14726,trt,U2OS,PAK1_WT.2,PAK1,72
986,TAORF_REFERENCE_SET,41755,J03,BRDN0000464939,trt,U2OS,PTEN_WT,PTEN,72
970,TAORF_REFERENCE_SET,41755,I11,negcon,trt,U2OS,LacZ_CTRL,LacZ,72
1824,TAORF_REFERENCE_SET,41757,M01,ccsbBroad304_06413,trt,U2OS,HSP90AA1_WT,HSP90AA1,72
1869,TAORF_REFERENCE_SET,41757,N22,ccsbBroad304_12357,trt,U2OS,RPTOR_WT,RPTOR,72


,Metadata_Plate,Metadata_Well,Metadata_pert_id,Metadata_pert_type,Metadata_cell_id,Metadata_ARP_ID,Metadata_pert_timepoint,Metadata_transcriptdb
1106,TA.OE015_A549_96H_X3_B19,G10,BRDN0000552709,trt,A549,DOC49.50.51.52.A,96,NM_001126112.2:c.476C>T
3993,TA.OE014_A549_96H_X2_B19,M06,BRDN0000560861,trt,A549,DOC45.46.47.48.A,96,NM_012289.3:c.425C>T
1782,TA.OE014_A549_96H_X7_B19,G01,TRCN0000478287,trt,A549,DOC45.46.47.48.A,96,NM_005676.4
3924,TA.OE015_A549_96H_X5_B19,A21,TRCN0000471252,trt,A549,DOC49.50.51.52.A,96,NM_022128.2
3655,TA.OE014_A549_96H_X8_B19,P10,BRDN0000560676,trt,A549,DOC45.46.47.48.A,96,NM_001654.4:c.641C>T


,Metadata_Plate_Map_Name,Metadata_Plate,Metadata_Well,Metadata_pert_id,Metadata_pert_type,Metadata_cell_id,Metadata_genesymbol_mutation,Metadata_genesymbol,Metadata_pert_timepoint
3453,DOC49.50.51.52,52654,B24,TRCN0000474810,trt,A549,NPDC1_WT.o,NPDC1,96
5709,DOC49.50.51.52,52654,N18,BRDN0000561664,trt,A549,TP53_WT.c,TP53,96
4915,DOC49.50.51.52,52674,J15,BRDN0000553388,trt,A549,MAP2K1_p.C121S,MAP2K1,96
1072,DOC45.46.47.48,52657,F15,BRDN0000560600,trt,A549,MDM2_WT.c,MDM2,96
5176,DOC49.50.51.52,52675,K24,BRDN0000553202,trt,A549,SMAD4_p.V492F,SMAD4,96


In [20]:
# Write to file
with open("dataset_samples.md", "w") as f:
    f.write(markdown_output)

print("Dataset samples have been written to dataset_samples.md")

Dataset samples have been written to dataset_samples.md


In [21]:
# Save processed datasets using same structure as input
for dataset_name, data_types in dataset_data.items():
    for data_type, data in data_types.items():
        # Mirror the input path structure but with processed data
        input_path = Path(dataset_paths[dataset_name][data_type])
        output_path = (
            Path("curated")
            / input_path.parent
            / input_path.name.replace(".csv.gz", ".parquet")
        )
        # Create the processed subdirectory if it doesn't exist
        output_path.parent.mkdir(exist_ok=True, parents=True)

        # # Save the data
        data.to_parquet(output_path, index=False)
        print(f"Saved {dataset_name} {data_type} data to {output_path}")

Saved LINCS-Pilot1 l1k data to curated/LINCS-Pilot1/L1000/replicate_level_l1k.parquet
Saved LINCS-Pilot1 cp data to curated/LINCS-Pilot1/CellPainting/replicate_level_cp_augmented.parquet
Saved CDRP-BBBC047-Bray l1k data to curated/CDRP-BBBC047-Bray/L1000/replicate_level_l1k.parquet
Saved CDRP-BBBC047-Bray cp data to curated/CDRP-BBBC047-Bray/CellPainting/replicate_level_cp_augmented.parquet
Saved TA-ORF-BBBC037-Rohban l1k data to curated/TA-ORF-BBBC037-Rohban/L1000/replicate_level_l1k.parquet
Saved TA-ORF-BBBC037-Rohban cp data to curated/TA-ORF-BBBC037-Rohban/CellPainting/replicate_level_cp_augmented.parquet
Saved LUAD-BBBC041-Caicedo l1k data to curated/LUAD-BBBC041-Caicedo/L1000/replicate_level_l1k.parquet
Saved LUAD-BBBC041-Caicedo cp data to curated/LUAD-BBBC041-Caicedo/CellPainting/replicate_level_cp_augmented.parquet
